# 01 · Setup: Unity Catalog foundations
**Demo:** Fashion Retail Inventory Intelligence (Siam Stride, a fictional footwear chain: 12 stores and 1 DC in Thailand)

This notebook creates the containers for the whole demo:
| Object | Name | Purpose |
|---|---|---|
| Catalog | `retail_demo` | Development catalog (top of the namespace) |
| Schema | `fashion_retail_inventory` | The whole demo: this is the unit we submit |
| Volume | `landing` | Raw files from source systems |

**Re-runnable:** every statement uses `IF NOT EXISTS`, so running it again is safe.
**Portable:** to deploy elsewhere, change only the catalog and schema names.


## 1. Catalog
The top level of Unity Catalog's three-level namespace: `catalog.schema.object`.
No storage location is given, so the catalog uses **managed storage**: Unity Catalog decides where the data is stored and manages it.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retail_demo
  COMMENT 'Development catalog for partner demos.';

## 2. Schema
The whole demo lives in one schema, matching how demos are organised in `partner_demo_catalog`.
Layers are shown by **table prefix**: `bronze_`, `silver_`, `gold_`, `mv_` (metric views), `ml_`, `fn_` (agent tools).

The comment is written for **business users**, because Genie and Genie Ontology read it.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_demo.fashion_retail_inventory
  COMMENT 'Fashion Retail Inventory Intelligence demo for Siam Stride (fictional footwear brand): 12 stores and 1 distribution centre (DC) in Thailand. Demand forecasting per store and SKU, and stock rebalancing between stores and from the DC. Layers are shown by table prefix: bronze_ (raw as delivered), silver_ (cleaned, validated), gold_ (business-ready dimensions, facts and KPIs), mv_ (metric views), ml_ (forecasts), fn_ (agent tools). All data is synthetic.';

## 3. Set the context
`USE` sets the default catalog and schema for the rest of this session, so everything after this can use short names like `landing`.
⚠️ If you detach or restart the notebook, **run this cell again** before running later cells on their own.


In [0]:
%sql
USE CATALOG retail_demo;
USE SCHEMA fashion_retail_inventory;
SELECT current_catalog(), current_schema();

## 4. Landing volume
A **Volume** is governed storage for *files* (CSV, JSON, images, PDFs). It is not a table.
Path: `/Volumes/retail_demo/fashion_retail_inventory/landing/`
Use one folder per source system and dataset (e.g. `pos/sales_lines/`), because Auto Loader reads one folder per table.


In [0]:
%sql
USE CATALOG retail_demo;
USE SCHEMA fashion_retail_inventory;
SELECT current_catalog(), current_schema();

%md
## 4. Landing volume
A **Volume** is governed storage for *files* (CSV, JSON, images, PDFs). It is not a table.
Path: `/Volumes/retail_demo/fashion_retail_inventory/landing/`
Use one folder per source system and dataset (e.g. `pos/sales_lines/`), because Auto Loader reads one folder per table.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS landing
  COMMENT 'Drop zone for source files from client systems (POS, ERP, merchandise planning). One folder per source system and dataset, e.g. pos/sales_lines/.';

## 5. Tags
**Comments** describe what something *means*. **Tags** *classify* it: they are searchable in Catalog Explorer and can be used in access policies.

In [0]:
%sql
ALTER SCHEMA fashion_retail_inventory SET TAGS (
  'project' = 'fashion_retail_inventory',
  'data_classification' = 'synthetic',
  'industry' = 'retail'
);

%md
## 6. Verify
Unity Catalog metadata can itself be queried with SQL, through `information_schema`.

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED fashion_retail_inventory;

In [0]:
%sql
DESCRIBE VOLUME landing;

In [0]:
%sql
SELECT schema_name, tag_name, tag_value
FROM retail_demo.information_schema.schema_tags
WHERE schema_name = 'fashion_retail_inventory';